# Apart from returns, what other variables would make a good ML model?

1) Risk & distribution stats (from prices)

What: realized vol/semivol, downside deviation, skew/kurtosis, rolling drawdowns, rolling betas/correlations, idiosyncratic vol.

Where: compute from Ticker.history() (OHLCV). yfinance provides split/dividend-adjusted series; you compute the stats. 
ranaroussi.github.io

2) Factor exposures / factor returns

What: exposures to market, size, value, momentum, profitability, investment; later map to tracking error and factor constraints.

Where: use Kenneth French Data Library for daily/monthly factor returns (FF3/FF5/MOM), then run rolling regressions to get exposures. 
mba.tuck.dartmouth.edu

Alt: WRDS mirror if you have access. 
WRDS

3) Fundamentals & quality ratios

What: valuation (PE, PB, EV/EBITDA), margins, growth, leverage, accruals, payout—useful for quality/low-risk tilts and constraint sets.

Where (yf): Ticker.get_income_stmt()/balance_sheet/cashflow/financials + trailing/quarterly/ttm helpers. 
ranaroussi.github.io

Alt: Nasdaq Data Link (Sharadar SF1) for point-in-time standardized fundamentals. 
Nasdaq Data Link
Nasdaq Data Docs

4) Analyst expectations & revisions

What: EPS/revenue estimates, surprise history, revisions, price targets; strong signals for short-horizon BO objectives.

Where (yf): get_earnings_estimate, get_revenue_estimate, get_eps_revisions, analyst_price_targets, get_earnings_history. 
ranaroussi.github.io

5) Events calendar

What: earnings dates, ex-dividends, splits; use to manage blackout windows or add event-aware penalties in the acquisition function.

Where (yf): get_earnings_dates() and calendar. 
ranaroussi.github.io
+1

6) Dividends & buybacks

What: dividend yield/growth, payout stability; shares-outstanding trends to proxy buybacks.

Where (yf): dividends, shares/get_shares_full, actions. 
ranaroussi.github.io

7) Options-implied metrics

What: 30-day IV, IV term structure/skew, put-call ratios—useful as regime/state variables and for downside-aware objectives (e.g., CVaR).

Where (yf): option_chain() for quotes; compute IV/term structure yourself. 
ranaroussi.github.io

Benchmarks: VIX as a market-wide implied vol proxy (also downloadable as ^VIX). Methodology: Cboe docs. 
CBOE
+1

8) Liquidity & microstructure

What: ADV, turnover, Amihud illiquidity, bid-ask/effective spread; helpful for turnover and transaction-cost penalties.

Where (yf): volume/turnover from history() (compute features). 
ranaroussi.github.io

Top-of-book spread/quotes: Polygon.io NBBO quotes; IEX Cloud also exposes real-time/historical quotes. 
Polygon
Postman
PublicAPI

9) Short interest & stock-loan/borrow costs

What: short interest %, days-to-cover, borrow fee/utilization; useful for squeeze/overcrowding risk and borrow-cost-aware objectives.

Where (official): FINRA bi-monthly short interest files + metadata/API guidance. 
FINRA
+2
FINRA
+2

Aggregated: Nasdaq short interest pages/API via Data Link; Polygon short-interest endpoint (mirrors FINRA). 
Nasdaq
Nasdaq Data Docs
Polygon

Borrow fees: Interactive Brokers SLB Rates (GUI/API). 
IBKR Guides
Interactive Brokers

10) Insider & institutional flows

What: insider transactions/holdings; institutional holders; changes can be priors or constraints.

Where (yf): insider_transactions, insider_purchases, insider_roster_holders, institutional_holders. 
ranaroussi.github.io

Primary source: SEC EDGAR APIs (submissions & XBRL). 
SEC
+1

11) ESG & controversies

What: aggregate ESG scores or governance red flags to build “soft” constraints or additional objectives.

Where (yf): sustainability (when available). 
ranaroussi.github.io

Alt: commercial providers (MSCI/Sustainalytics/Refinitiv) if you need coverage/consistency.

12) Macro & rates (for risk-free, constraints, & regimes)

What: risk-free (1M–10Y UST), term spreads, credit spreads, inflation, unemployment, PMI; use as context features or regime switches.

Where: FRED API (and fredapi Python wrapper). 
FRED
GitHub

In [10]:
import pandas as pd
import yfinance as yf
import warnings

warnings.filterwarnings("ignore")
pd.options.display.float_format = '{:.4%}'.format

# Date range
start = '2023-08-18'
end = '2025-08-19'
interval = "1wk"

# Tickers of assets 
assets = ['AVEM', 'XMMO', 'ESGD', 'BYLD', 'ISCF', 'HYEM', 'VWOB', 'VNQI', 'VNQ', 'SNPE', 'AVES', 'IEF', 'VBK'] # , 'AGEPX', 'SWSSX', 'SWVXX' mutual funds 
assets.sort()
raw_data = yf.download(assets, start=start, end=end, auto_adjust=False, interval=interval)
data = raw_data.loc[:, ('Adj Close', slice(None))]
data.columns = assets
data

[*********************100%***********************]  13 of 13 completed


,AVEM,AVES,BYLD,ESGD,HYEM,IEF,ISCF,SNPE,VBK,VNQ,VNQI,VWOB,XMMO
Date,,,,,,,,,,,,,
2023-08-14,4949.4450%,3953.6819%,1944.5383%,6591.2048%,1566.7085%,8716.7847%,2778.7479%,3898.5138%,21800.8331%,7419.4946%,3603.8891%,5345.5383%,7986.3579%
2023-08-21,5029.3350%,3997.8783%,1953.0949%,6630.6946%,1569.3452%,8725.1686%,2785.1780%,3936.5528%,21934.1080%,7465.6128%,3638.6471%,5397.7547%,8024.8962%
2023-08-28,5137.4226%,4083.5072%,1963.0026%,6757.6302%,1580.7676%,8752.1774%,2847.6425%,4043.8419%,22881.8405%,7611.3464%,3709.9934%,5404.8351%,8228.4584%
2023-09-04,5051.8925%,4041.8900%,1951.2932%,6661.7233%,1588.7103%,8730.6923%,2787.9339%,3990.1974%,22197.6974%,7524.6437%,3688.9553%,5407.1220%,8003.1555%
2023-09-11,5098.8869%,4079.8244%,1956.6715%,6751.0483%,1592.2447%,8691.4772%,2799.8756%,3970.6902%,21964.7110%,7545.8580%,3681.6376%,5388.4464%,8024.8962%
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-21,7052.9999%,5515.0002%,2243.9558%,9037.9997%,1977.4057%,9443.1519%,3977.9999%,5709.0000%,28717.0013%,9173.0003%,4634.0000%,6501.4648%,13211.9995%
2025-07-28,6854.0001%,5354.9999%,2256.4001%,8705.0003%,1973.4291%,9536.8462%,3871.9002%,5588.0001%,27773.0011%,8891.9998%,4550.0000%,6493.5120%,12883.0002%
2025-08-04,7055.0003%,5504.0001%,2249.9290%,8997.0001%,1988.9999%,9537.9997%,3988.9999%,5738.9999%,27894.0002%,8905.9998%,4688.9999%,6576.9997%,13013.0005%


In [11]:
# Calculating returns

Y = data[assets].pct_change().dropna()

display(Y.head())

,AVEM,AVES,BYLD,ESGD,HYEM,IEF,ISCF,SNPE,VBK,VNQ,VNQI,VWOB,XMMO
Date,,,,,,,,,,,,,
2023-08-21,1.6141%,1.1179%,0.4400%,0.5991%,0.1683%,0.0962%,0.2314%,0.9757%,0.6113%,0.6216%,0.9645%,0.9768%,0.4826%
2023-08-28,2.1491%,2.1419%,0.5073%,1.9144%,0.7278%,0.3096%,2.2427%,2.7255%,4.3208%,1.9521%,1.9608%,0.1312%,2.5366%
2023-09-04,-1.6648%,-1.0192%,-0.5965%,-1.4192%,0.5025%,-0.2455%,-2.0968%,-1.3266%,-2.9899%,-1.1391%,-0.5671%,0.0423%,-2.7381%
2023-09-11,0.9302%,0.9385%,0.2756%,1.3409%,0.2225%,-0.4492%,0.4283%,-0.4889%,-1.0496%,0.2819%,-0.1984%,-0.3454%,0.2717%
2023-09-18,-1.5484%,-1.1848%,-0.4067%,-2.1309%,-0.3330%,-0.6982%,-1.8045%,-2.9105%,-4.2878%,-5.3416%,-1.4410%,-0.7592%,-2.5366%


In [12]:
# get etf categories 
categories = {}

for asset in assets:
    ticker = yf.Ticker(asset)
    info = ticker.info
    category = info.get('category', 'Unknown')
    categories[asset] = category
categories

{'AVEM': 'Diversified Emerging Mkts',
 'AVES': 'Diversified Emerging Mkts',
 'BYLD': 'Multisector Bond',
 'ESGD': 'Foreign Large Blend',
 'HYEM': 'Emerging Markets Bond',
 'IEF': 'Long Government',
 'ISCF': 'Foreign Small/Mid Blend',
 'SNPE': 'Large Blend',
 'VBK': 'Small Growth',
 'VNQ': 'Real Estate',
 'VNQI': 'Global Real Estate',
 'VWOB': 'Emerging Markets Bond',
 'XMMO': 'Mid-Cap Blend'}

In [ ]:
fundamentals = {}

asset_class = {'AVEM': 'Equity',
 'AVES': 'Equity',
 'BYLD': 'Fixed Income',
 'ESGD': 'Equity',
 'HYEM': 'Fixed Income',
 'IEF': 'Fixed Income',
 'ISCF': 'Equity',
 'SNPE': 'Equity',
 'VBK': 'Equity',
 'VNQ': 'Real Estate',
 'VNQI': 'Real Estate',
 'VWOB': 'Fixed Income',
 'XMMO': 'Equity'}

region = {'AVEM': 'International',
 'AVES': 'International',
 'BYLD': 'US',
 'ESGD': 'International',
 'HYEM': 'International',
 'IEF': 'US',
 'ISCF': 'International',
 'SNPE': 'US',
 'VBK': 'US',
 'VNQ': 'Real Estate',
 'VNQI': 'International',
 'VWOB': 'International',
 'XMMO': 'US'}

region = {}

In [15]:
etf = yf.Ticker('AVEM')

fd = etf.funds_data              # or: etf.get_funds_data()
fd.sector_weightings             # -> dict of sector -> weight (0–1)
fd.top_holdings                  # -> DataFrame of top holdings (symbol, name, weight)
fd.equity_holdings               # -> summary ratios (P/E, P/B, etc.)
fd.asset_classes                 # -> {'stocks': ..., 'bonds': ...}


{'cashPosition': 0.0014,
 'stockPosition': 0.9984,
 'bondPosition': 0.0,
 'preferredPosition': 0.0002,
 'convertiblePosition': 0.0,
 'otherPosition': 0.0}

In [14]:
fd.sector_weightings 

{'realestate': 0.0194,
 'consumer_cyclical': 0.0851,
 'basic_materials': 0.049200002,
 'consumer_defensive': 0.0771,
 'technology': 0.10569999,
 'communication_services': 0.0551,
 'financial_services': 0.24780001,
 'utilities': 0.033099998,
 'industrials': 0.1824,
 'energy': 0.0392,
 'healthcare': 0.1058}

# Approach to building a model:

1. Create long dataset where each row is one asset-time observation.
2. Variables to include would be 
    - time
    - asset id
    -features for that assset at that date
    -features across assets at that data (macrorates, inflation, etc.).
    -maybe lags or rolling regressions
3. Target variable: y_excess_lead?? is the future excess total return for each asset over a fixed horizon h

# What exactly is the target?

	•	Use Adjusted Close from Yahoo Finance (yfinance) so dividends and splits are included in r_{i,t\to t+h}.
	•	The risk-free piece should be accrued over the same dates as the asset return.

### Create a long dataframe with y_excess_lead as the outptu variable

In [ ]:
# ==== CONFIG ====
TICKERS = ['AVEM', 'XMMO', 'ESGD', 'BYLD', 'ISCF', 'HYEM', 'VWOB', 'VNQI', 'VNQ', 'SNPE', 'AVES', 'IEF', 'VBK']
START = "2023-01-01"      # adjust as needed
END = None                # None = up to today
HORIZON = "weekly"        # options: "weekly", "monthly", or "fixed"
FIXED_H_DAYS = 5          # only used if HORIZON == "fixed"

# ==== CODE ====
import numpy as np
import pandas as pd
import yfinance as yf

def fetch_prices(tickers, start, end=None):
    """
    Returns a tidy df with MultiIndex (date, ticker) and column 'Adj Close'.
    Adjusted Close includes dividends & splits => total-return compatible.
    """
    px = yf.download(
        tickers=tickers, start=start, end=end, interval="1d",
        group_by="ticker", auto_adjust=False, progress=False
    )
    # Normalize shape across yfinance versions
    if isinstance(px.columns, pd.MultiIndex):
        # Multi-index columns: level 0 = ticker, level 1 = field
        adj = []
        for tk in tickers:
            if (tk, 'Adj Close') in px.columns:
                s = px[(tk, 'Adj Close')].rename(tk)
                adj.append(s)
        adj = pd.concat(adj, axis=1)
    else:
        # Single-index columns: use 'Adj Close' directly (single ticker)
        adj = px[['Adj Close']].rename(columns={'Adj Close': tickers[0]})
    adj = adj.dropna(how="all")
    adj.index = pd.to_datetime(adj.index)
    return adj

def fetch_rf_daily(start, end=None):
    """
    Fetch ^IRX (13-week T-bill), robust to MultiIndex columns and auto_adjust True/False.
    Returns a daily series of *continuous* daily rate ≈ (annual_fraction / 252).
    """
    rf = yf.download("^IRX", start=start, end=end, interval="1d", progress=False, auto_adjust=False)

    if rf.empty:
        raise RuntimeError("Could not download ^IRX. Try expanding the date range.")

    # Handle both single-index and MultiIndex column shapes
    if isinstance(rf.columns, pd.MultiIndex):
        col = None
        candidates = [
            ("Adj Close", "^IRX"), ("Close", "^IRX"),
            ("^IRX", "Adj Close"), ("^IRX", "Close")
        ]
        for c in candidates:
            if c in rf.columns:
                col = rf.loc[:, c].astype(float)
                break
        if col is None:
            # Fallback: slice by level name, then pick the first column
            if "Adj Close" in rf.columns.get_level_values(0):
                col = rf.xs("Adj Close", level=0, axis=1).iloc[:, 0].astype(float)
            else:
                col = rf.xs("Close", level=0, axis=1).iloc[:, 0].astype(float)
    else:
        if "Adj Close" in rf.columns:
            col = rf["Adj Close"].astype(float)
        elif "Close" in rf.columns:
            col = rf["Close"].astype(float)
        else:
            raise KeyError(f"^IRX: Neither 'Adj Close' nor 'Close' in columns: {rf.columns.tolist()}")

    # Convert annualized percent -> fraction
    ann_frac = col / 100.0
    rf_daily_cont = ann_frac / 252.0
    rf_daily_cont.index = pd.to_datetime(rf_daily_cont.index)

    # Fill to business-day grid and forward-fill small gaps
    all_bd = pd.date_range(rf_daily_cont.index.min(), rf_daily_cont.index.max(), freq="B")
    rf_daily_cont = rf_daily_cont.reindex(all_bd).ffill()
    rf_daily_cont.name = "rf_daily_cont"
    return rf_daily_cont

def compute_excess_future_return_calendar(adj_close: pd.Series,
                                          rf_daily_cont: pd.Series,
                                          freq: str = "W-FRI"):
    """
    Calendar-aligned future excess return for one asset.
    y_excess = exp( log(P_{t1}) - log(P_t) - ∑_{(t,t1]} rf_daily_cont ) - 1
    Returns a Series indexed by the start-of-period date t.
    """
    # Resample to period-end closes
    px_period = adj_close.resample(freq).last().dropna()
    period_end = px_period.index
    px_fwd = px_period.shift(-1)

    # Price log-return for t -> t1
    log_r_price = (np.log(px_fwd) - np.log(px_period)).iloc[:-1]

    # Integrate RF over (t, t1] by summing continuous daily rates
    rf_log = []
    for t, t1 in zip(period_end[:-1], period_end[1:]):
        days = rf_daily_cont.loc[(rf_daily_cont.index > t) & (rf_daily_cont.index <= t1)]
        rf_log.append(days.sum() if not days.empty else 0.0)
    rf_log = pd.Series(rf_log, index=log_r_price.index)

    # Use numpy arrays to avoid dtype gotchas, then rebuild Series
    delta = (log_r_price.values - rf_log.values)
    y_excess_vals = np.exp(delta) - 1.0
    y_excess = pd.Series(y_excess_vals, index=log_r_price.index, name="y_excess_lead")
    return y_excess

def compute_excess_future_return_fixed(adj_close: pd.Series,
                                       rf_daily_cont: pd.Series,
                                       h_days: int = 5):
    """
    Fixed trading-day horizon (t -> t+h).
    y_excess = exp( log(P_{t+h}) - log(P_t) - sum_{k=0..h-1} rf_daily_cont[t+k] ) - 1
    """
    # Ensure we have RF on the same business-day index
    s = adj_close.dropna()
    idx = s.index
    rf_on_px = rf_daily_cont.reindex(idx).ffill()

    log_p = np.log(s)
    log_r_price = log_p.shift(-h_days * -1) - log_p  # log(P_{t+h}) - log(P_t); shift(-h) forward
    # Sum RF daily cont rates over next h days, aligned at t
    rf_log = rf_on_px.shift(-h_days + 1).rolling(h_days).sum()  # sum of future h days
    # Target
    y_excess = np.exp(log_r_price - rf_log) - 1.0
    y_excess = y_excess.iloc[:-h_days]  # drop last h because of lookahead
    y_excess.name = "y_excess_lead"
    return y_excess

def build_long_panel(tickers, start, end=None, horizon="weekly", fixed_h_days=5):
    prices = fetch_prices(tickers, start, end)
    rf_daily_cont = fetch_rf_daily(start, end)

    # Choose calendar frequency
    if horizon.lower() == "weekly":
        freq = "W-FRI"
        compute_fn = lambda s: compute_excess_future_return_calendar(s, rf_daily_cont, freq=freq)
    elif horizon.lower() == "monthly":
        freq = "M"
        compute_fn = lambda s: compute_excess_future_return_calendar(s, rf_daily_cont, freq=freq)
    elif horizon.lower() == "fixed":
        compute_fn = lambda s: compute_excess_future_return_fixed(s, rf_daily_cont, h_days=fixed_h_days)
    else:
        raise ValueError("horizon must be one of {'weekly','monthly','fixed'}")

    rows = []
    for tk in prices.columns:
        try:
            y_excess = compute_fn(prices[tk].dropna())
            df_tk = y_excess.to_frame()
            df_tk["asset_id"] = tk
            df_tk = df_tk.rename_axis("date").reset_index()[["date", "asset_id", "y_excess_lead"]]
            rows.append(df_tk)
        except Exception as e:
            print(f"[WARN] Skipping {tk}: {e}")

    panel = pd.concat(rows, axis=0).sort_values(["date", "asset_id"]).reset_index(drop=True)
    return panel

In [8]:
panel = build_long_panel(TICKERS, START, END, horizon=HORIZON, fixed_h_days=FIXED_H_DAYS)

### How to interpret y_excess_lead?
- Positive values → asset outperformed cash over that horizon.
- Negative values → asset underperformed cash.
- Magnitude gives you the scale (e.g. 0.02 = +2% excess over the week).

For example, if the excess lead of ISCF on 2023-01-06 was 0.0303 
(3.03%), it means ISCF outperformed cash by 3.03% over the next week.

In [9]:
panel



,date,asset_id,y_excess_lead
0,2023-01-06,AVEM,2.8878%
1,2023-01-06,BYLD,0.5377%
2,2023-01-06,ESGD,3.5247%
3,2023-01-06,HYEM,0.4976%
4,2023-01-06,ISCF,3.0306%
...,...,...,...
1365,2025-08-15,SNPE,-0.9462%
1366,2025-08-15,VNQ,1.0991%
1367,2025-08-15,VNQI,0.6710%
1368,2025-08-15,VWOB,-0.0339%


# Adding context variables

1. **Price-based / Technical features**

These come straight from the ETF’s own return history:

-   Lagged returns (1-week, 2-week, 4-week, etc.)
-   Momentum: rolling cumulative returns over 3m, 6m, 12m.
-   Volatility measures: rolling standard deviation (e.g. 4w, 12w).
-   Drawdown: max decline from recent peak.
-   Skew/kurtosis of rolling returns.
-   Moving averages (20-day, 50-day, 200-day) and crossovers.
-   RSI / stochastic oscillators (basic technicals).

2. **Cross-sectional / Fundamental ETF descriptors**

Even though ETFs don’t have classic accounting statements, they inherit factor exposures:

-   Asset class tags (equity, fixed income, REIT, emerging markets, etc.).
-   Region/country exposures (developed, EM, Asia, US).
-   Style (value vs growth, small vs large cap).
-   Duration / credit exposure for bond ETFs (BYLD, HYEM, VWOB).
-   Sector tilts (VNQ/VNQI = real estate). 

These can be encoded as categorical one-hot vectors or embeddings.

3. **Market-wide / Macro features**

All ETFs are affected by common risk factors:

-   Equity market index returns (S&P 500, MSCI World).
-   Bond yields: 2y, 10y Treasury yields, yield curve slope.
-   Credit spreads: investment grade vs HY spread.
-   FX indices (USD index).
-   Commodities: oil, gold.
-   Volatility index (VIX).
-   Inflation surprise indices, unemployment, PMI (if you want to bring in economic releases).

These are typically aligned to the same horizon as your ETF panel (weekly).


4. **Calendar / Regime features**

Markets behave differently under certain regimes:

-   Month of year / quarter of year (seasonality effects).

-   Week-of-month / day-of-week (if you go daily).

-   Recession vs expansion flags (NBER or proxy).

-   Fed meeting weeks, earnings season flags.

-   Volatility regime (low/high VIX).

# Price based features

**Note**
All features are computed on resampled weekly closes to match your Friday→Friday target and avoid look-ahead.

-   Momentum features (e.g., mom_13w) are cumulative: 
(1 + 𝑟) ^ 13 −1

-   Volatility uses weekly returns’ std, not price std.

-   RSI uses Wilder smoothing on weekly price changes (rsi_14w).

-   Expect NaNs at the beginning for windows like 26 or 52 weeks—normal. Inner-joining with the target naturally trims the very early warm-up period.


### 📊 Price-Based Feature Definitions

| Feature       | Definition                                                                                   | Lookback Window            |
|---------------|-----------------------------------------------------------------------------------------------|----------------------------|
| **ret_1w**    | Simple return over the past 1 week.                                                           | 1 week                     |
| **ret_2w**    | Cumulative return over the past 2 weeks.                                                      | 2 weeks                    |
| **ret_4w**    | Cumulative return over the past 4 weeks (~1 month).                                           | 4 weeks                    |
| **mom_13w**   | Momentum: cumulative return over the past 13 weeks (~quarter).                                | 13 weeks (~3 months)       |
| **mom_26w**   | Momentum: cumulative return over the past 26 weeks (~half year).                              | 26 weeks (~6 months)       |
| **mom_52w**   | Momentum: cumulative return over the past 52 weeks (~1 year).                                 | 52 weeks (~12 months)      |
| **vol_4w**    | Rolling standard deviation of weekly returns.                                                 | 4 weeks                    |
| **vol_12w**   | Rolling standard deviation of weekly returns.                                                 | 12 weeks (~3 months)       |
| **vol_26w**   | Rolling standard deviation of weekly returns.                                                 | 26 weeks (~6 months)       |
| **dd_26w**    | Current drawdown relative to rolling maximum price (how far below recent peak).               | 26 weeks (~6 months)       |
| **ma_4w**     | 4-week simple moving average of price.                                                        | 4 weeks                    |
| **ma_12w**    | 12-week simple moving average of price.                                                       | 12 weeks (~3 months)       |
| **ma_26w**    | 26-week simple moving average of price.                                                       | 26 weeks (~6 months)       |
| **px_ma4_gap**  | % difference between price and its 4-week moving average.                                    | 4 weeks                    |
| **px_ma12_gap** | % difference between price and its 12-week moving average.                                   | 12 weeks (~3 months)       |
| **px_ma26_gap** | % difference between price and its 26-week moving average.                                   | 26 weeks (~6 months)       |
| **skew_12w**  | Skewness of weekly returns distribution (asymmetry).                                          | 12 weeks (~3 months)       |
| **kurt_12w**  | Excess kurtosis of weekly returns distribution (fat tails vs. normal).                        | 12 weeks (~3 months)       |
| **rsi_14w**   | Relative Strength Index (RSI) based on weekly closes. >70 = “overbought”, <30 = “oversold”.   | 14 weeks                   |

In [16]:
# ========================
# Price-based feature pack
# ========================

def _rolling_cumret(r: pd.Series, win: int) -> pd.Series:
    """(1+r).rolling(win).apply(prod)-1, stable for NaNs."""
    return (1.0 + r).rolling(window=win, min_periods=win).apply(np.prod, raw=True) - 1.0

def _rsi(prices: pd.Series, period: int = 14) -> pd.Series:
    """Wilder-style RSI on the *resampled* price series (weekly here)."""
    delta = prices.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    # Wilder's smoothing
    avg_gain = gain.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi

def _weekly_price_table(prices_wide: pd.DataFrame, freq: str = "W-FRI") -> pd.DataFrame:
    """Resample to weekly (Friday) closes, consistent with y_excess_lead."""
    px_w = prices_wide.resample(freq).last().dropna(how="all")
    return px_w

def _weekly_return_table(px_w: pd.DataFrame) -> pd.DataFrame:
    """Simple weekly returns from resampled prices."""
    return px_w.pct_change()

def compute_price_features_for_one(
    px_w: pd.Series,  # weekly price series for one ETF
) -> pd.DataFrame:
    """
    Returns a DataFrame indexed by date with all price-based features for one asset.
    All features use ONLY data up to 'date' (no leakage).
    """
    s = px_w.dropna()
    ret = s.pct_change()

    # --- Last-week and multi-week momentum (simple cumulative returns) ---
    feat = pd.DataFrame(index=s.index)
    feat["ret_1w"]  = ret                                  # last week return
    feat["ret_2w"]  = _rolling_cumret(ret, 2)
    feat["ret_4w"]  = _rolling_cumret(ret, 4)
    feat["mom_13w"] = _rolling_cumret(ret, 13)             # ~3m
    feat["mom_26w"] = _rolling_cumret(ret, 26)             # ~6m
    feat["mom_52w"] = _rolling_cumret(ret, 52)             # ~12m

    # --- Volatility (std of weekly returns) ---
    feat["vol_4w"]  = ret.rolling(4,  min_periods=4).std()
    feat["vol_12w"] = ret.rolling(12, min_periods=12).std()
    feat["vol_26w"] = ret.rolling(26, min_periods=26).std()

    # --- Drawdown (relative to rolling max) ---
    rolling_max_26 = s.rolling(26, min_periods=26).max()
    feat["dd_26w"] = (s / rolling_max_26) - 1.0

    # --- Moving averages & price gaps ---
    ma4  = s.rolling(4,  min_periods=4).mean()
    ma12 = s.rolling(12, min_periods=12).mean()
    ma26 = s.rolling(26, min_periods=26).mean()
    feat["ma_4w"]   = ma4
    feat["ma_12w"]  = ma12
    feat["ma_26w"]  = ma26
    feat["px_ma4_gap"]  = (s / ma4)  - 1.0
    feat["px_ma12_gap"] = (s / ma12) - 1.0
    feat["px_ma26_gap"] = (s / ma26) - 1.0

    # --- Higher moments of returns ---
    feat["skew_12w"] = ret.rolling(12, min_periods=12).skew()
    # Fisher's definition (excess kurtosis): pandas' .kurt() already returns Fisher by default
    feat["kurt_12w"] = ret.rolling(12, min_periods=12).kurt()

    # --- RSI on weekly prices ---
    feat["rsi_14w"] = _rsi(s, period=14)

    return feat

def build_price_features_panel(
    tickers, start, end=None, horizon="weekly"
) -> pd.DataFrame:
    """
    Builds a LONG panel of price-based features aligned to your weekly calendar.
    Columns: date, asset_id, <features...>
    """
    # Use the same price fetcher you already have
    prices = fetch_prices(tickers, start, end)

    # Pick calendar matching your target
    if horizon.lower() == "weekly":
        freq = "W-FRI"
    elif horizon.lower() == "monthly":
        freq = "M"
    else:
        # If you use fixed daily horizons, you could compute daily features instead.
        # For now we match your weekly setup.
        freq = "W-FRI"

    px_w = _weekly_price_table(prices, freq=freq)

    # Compute features per asset and stack long
    long_rows = []
    for tk in px_w.columns:
        f = compute_price_features_for_one(px_w[tk])
        f = f.rename_axis("date").reset_index()
        f.insert(1, "asset_id", tk)
        long_rows.append(f)

    features_long = pd.concat(long_rows, axis=0).sort_values(["date", "asset_id"]).reset_index(drop=True)
    return features_long

def build_dataset_with_features(
    tickers, start, end=None, horizon="weekly", fixed_h_days=5
) -> pd.DataFrame:
    """
    Convenience wrapper: builds your target panel and merges price features.
    Returns a single LONG dataframe ready for modeling.
    """
    target_panel = build_long_panel(tickers, start, end, horizon=horizon, fixed_h_days=fixed_h_days)
    feat_panel   = build_price_features_panel(tickers, start, end, horizon=horizon)

    # Inner-join: keeps only dates/assets present in BOTH (prevents leakage/misalignment)
    df = pd.merge(target_panel, feat_panel, on=["date", "asset_id"], how="inner")
    # Optional: drop rows that still have too many NaNs (from warm-up windows)
    # df = df.dropna(subset=["ret_1w","vol_4w","mom_13w"])  # minimal viable set
    return df


In [ ]:
# 1) Target only (what you already do)
panel_y = build_long_panel(TICKERS, START, END, horizon=HORIZON, fixed_h_days=FIXED_H_DAYS)

# 2) Price-based features
panel_x_price = build_price_features_panel(TICKERS, START, END, horizon=HORIZON)

# 3) One merged dataset (X + y)
dataset = build_dataset_with_features(TICKERS, START, END, horizon=HORIZON, fixed_h_days=FIXED_H_DAYS)
dataset.head()

        date asset_id  y_excess_lead  ret_1w  ret_2w  ret_4w  mom_13w  \
0 2023-01-06     AVEM        2.8878%     NaN     NaN     NaN      NaN   
1 2023-01-06     BYLD        0.5377%     NaN     NaN     NaN      NaN   
2 2023-01-06     ESGD        3.5247%     NaN     NaN     NaN      NaN   
3 2023-01-06     HYEM        0.4976%     NaN     NaN     NaN      NaN   
4 2023-01-06     ISCF        3.0306%     NaN     NaN     NaN      NaN   

   mom_26w  mom_52w  vol_4w  ...  dd_26w  ma_4w  ma_12w  ma_26w  px_ma4_gap  \
0      NaN      NaN     NaN  ...     NaN    NaN     NaN     NaN         NaN   
1      NaN      NaN     NaN  ...     NaN    NaN     NaN     NaN         NaN   
2      NaN      NaN     NaN  ...     NaN    NaN     NaN     NaN         NaN   
3      NaN      NaN     NaN  ...     NaN    NaN     NaN     NaN         NaN   
4      NaN      NaN     NaN  ...     NaN    NaN     NaN     NaN         NaN   

   px_ma12_gap  px_ma26_gap  skew_12w  kurt_12w  rsi_14w  
0          NaN          NaN

Next steps:
-   Add style/factor exposures (from ETF characteristics).
-   Add market/macro features
-   Calendar / Regime features